# 08 - Modelo final e teste cego de 2025

> **NAO EXECUTE ESTE NOTEBOOK ANTES DE CONCLUIR 07b E 07c.**

Este notebook e deliberadamente protegido contra uma abertura acidental de 2025. A primeira configuracao exige que voce escreva explicitamente qual arquitetura foi congelada depois da comparacao em 2024.

Opcoes suportadas:

- `xgboost_radar` - referencia tabular atual;
- `TCN` - rede temporal multi-head;
- `GRU` - rede temporal multi-head.

A regra e:

1. escolher arquitetura, lookback, hiperparametros e limiares usando somente 2015-2024;
2. congelar essas escolhas;
3. treinar a versao final usando os dados pre-2025 permitidos;
4. abrir 2025 uma unica vez para avaliacao final.

Rodar novamente o mesmo modelo para regenerar arquivos nao e problema. O problema e olhar 2025, mudar decisoes por causa dele e chama-lo novamente de "teste cego".

In [ ]:
# ============================================================
# TRAVA METODOLOGICA
# ============================================================
FINAL_MODEL_KIND = None
# Depois de ler o relatorio 07c, troque por UMA destas opcoes:
# FINAL_MODEL_KIND = "xgboost_radar"
# FINAL_MODEL_KIND = "TCN"
# FINAL_MODEL_KIND = "GRU"

if FINAL_MODEL_KIND is None:
    raise RuntimeError(
        "Teste cego bloqueado. Execute 07b e 07c, escolha a arquitetura final e somente entao "
        "edite FINAL_MODEL_KIND nesta celula."
    )

In [ ]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
CWD=Path.cwd().resolve(); ROOT=CWD.parent if CWD.name.lower()=="notebooks" else CWD
PROCESSED=ROOT/"data"/"processed"; MODELS=ROOT/"models"; OUTPUTS=ROOT/"outputs"
TABLES=OUTPUTS/"tables"; FIGURES=OUTPUTS/"figures"; PREDICTIONS=OUTPUTS/"predictions"
for p in [MODELS,TABLES,FIGURES,PREDICTIONS]: p.mkdir(parents=True,exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))

from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Arquitetura congelada:",FINAL_MODEL_KIND)

## 1. Branch A - XGBoost + radar

Se `FINAL_MODEL_KIND="xgboost_radar"`, o notebook usa todas as features tabulares e de radar. O numero de arvores e congelado a partir do melhor `best_iteration` encontrado antes de 2025.

In [ ]:
blind_results=None
alert_probabilities=None
alert_thresholds=None

if FINAL_MODEL_KIND == "xgboost_radar":
    from xgboost import XGBRegressor

    path=PROCESSED/"features_causal_radar.parquet"
    if not path.exists():
        raise FileNotFoundError("Rode o notebook 06 v5 primeiro: features_causal_radar.parquet nao existe.")
    df=pd.read_parquet(path).sort_index()
    target="target_delta_120m"
    feature_cols=[c for c in df.columns if c!="split" and not c.startswith("target_")]
    fit=df[df["split"].isin(["train","validation","test_2024"])].dropna(subset=["stage_now",target])
    blind=df[df["split"]=="blind_2025"].dropna(subset=["stage_now",target])

    dev=XGBRegressor(); dev.load_model(MODELS/"xgb_radar_dev_120m.json")
    try:
        best_iteration=int(dev.get_booster().attr("best_iteration"))+1
    except Exception:
        try: best_iteration=int(dev.best_iteration)+1
        except Exception: best_iteration=600
    print("Numero de arvores congelado:",best_iteration)

    final_model=XGBRegressor(
        objective="reg:squarederror",eval_metric="rmse",tree_method="hist",
        n_estimators=best_iteration,learning_rate=0.035,max_depth=6,min_child_weight=20,
        subsample=0.85,colsample_bytree=0.80,reg_lambda=5.0,max_bin=128,n_jobs=-1,random_state=42,
    )
    final_model.fit(fit[feature_cols],fit[target])
    pred=final_model.predict(blind[feature_cols])
    blind_results=blind[["stage_now",target]].copy()
    blind_results["pred_delta_120m"]=pred
    blind_results["prediction_time"]=blind_results.index
    final_model.save_model(MODELS/"final_xgboost_radar_120m.json")

## 2. Branch B - TCN/GRU

Para uma rede temporal, o pre-processador e refitado usando apenas os dados pre-2025. O numero final de epocas e **fixado pelo melhor epoch de validacao do notebook 07b**, em vez de olhar 2025 para decidir quando parar.

In [ ]:
if FINAL_MODEL_KIND in ["TCN","GRU"]:
    import torch
    from torch.utils.data import DataLoader
    from utils.temporal_models import (
        TemporalPreprocessor,SequenceDataset,TCNMultiTask,GRUMultiTask,TrainingConfig,
        build_temporal_frame,valid_sequence_indices,fit_fixed_epochs,predict_model,choose_device,
    )

    master=pd.read_parquet(PROCESSED/"master_base.parquet").sort_index()
    features=pd.read_parquet(PROCESSED/"features_causal.parquet").sort_index()
    radar=pd.read_parquet(PROCESSED/"radar_features_basic.parquet").sort_index()
    if radar.index.has_duplicates: radar=radar.groupby(level=0).mean(numeric_only=True).sort_index()
    raw,groups=build_temporal_frame(master,radar=radar)
    target=features["target_delta_120m"].reindex(raw.index)
    split=features["split"].reindex(raw.index)

    registry=json.loads((MODELS/"neural_registry_v6.json").read_text(encoding="utf-8"))
    lookback=int(registry["lookback_steps"])
    fit_mask=split.isin(["train","validation","test_2024"]) & target.notna()
    pre=TemporalPreprocessor(groups,stage_ffill_limit=6).fit(raw,fit_mask,target)
    scaled=pre.transform(raw); matrix=scaled.to_numpy("float32"); yraw=target.to_numpy("float32"); yscaled=pre.transform_target(yraw)
    pre.state.to_json(MODELS/f"final_{FINAL_MODEL_KIND.lower()}_preprocessor.json")

    split_final=split.copy(); split_final.loc[split_final.isin(["train","validation","test_2024"]) ]="fit_final"
    idx_fit=valid_sequence_indices(raw.index,split_final,target,"fit_final",lookback,stride=2)
    idx_blind=valid_sequence_indices(raw.index,split_final,target,"blind_2025",lookback,stride=1)
    fit_ds=SequenceDataset(matrix,yscaled,yraw,idx_fit,lookback); blind_ds=SequenceDataset(matrix,yscaled,yraw,idx_blind,lookback)
    fit_loader=DataLoader(fit_ds,batch_size=512,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
    blind_loader=DataLoader(blind_ds,batch_size=512,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())

    hist_path=TABLES/f"{FINAL_MODEL_KIND.lower()}_training_history_v6.csv"
    hist=pd.read_csv(hist_path)
    fixed_epochs=int(hist.loc[hist["valid_loss"].idxmin(),"epoch"])
    print("Epocas finais congeladas:",fixed_epochs)
    input_dim=matrix.shape[1]
    if FINAL_MODEL_KIND=="TCN": model=TCNMultiTask(input_dim,channels=(64,64,96,96),kernel_size=3,dropout=.15)
    else: model=GRUMultiTask(input_dim,hidden_dim=96,num_layers=2,dropout=.15)
    cfg=TrainingConfig(model_name=FINAL_MODEL_KIND,lookback_steps=lookback,batch_size=512,max_epochs=fixed_epochs,patience=0,seed=42)
    fit_hist=fit_fixed_epochs(model,fit_loader,yraw[idx_fit],cfg,fixed_epochs,MODELS/f"final_{FINAL_MODEL_KIND.lower()}_120m.pt",device=choose_device())
    pred_df=predict_model(model,blind_loader,pre,raw.index,device=choose_device())
    pred_df=pred_df.set_index("prediction_time")
    blind_results=pred_df[["target_delta_120m","pred_delta_120m"]].copy()
    blind_results["stage_now"]=master["stage_413"].reindex(blind_results.index)
    blind_results["prediction_time"]=blind_results.index
    alert_probabilities=pred_df[["p_ge_1m","p_ge_2m","p_ge_3m"]].copy()

    th=pd.read_csv(TABLES/"neural_probability_thresholds_2023_v6.csv")
    th=th[th["model"]==FINAL_MODEL_KIND].set_index("event_threshold_m")
    alert_thresholds={1.0:float(th.loc[1.0,"threshold"]),2.0:float(th.loc[2.0,"threshold"]),3.0:float(th.loc[3.0,"threshold"])}

## 3. Abertura do holdout 2025

A partir daqui, o valor real de 2025 e usado **somente para medir o modelo congelado**.

In [ ]:
if blind_results is None:
    raise RuntimeError("Arquitetura nao executada.")

y=blind_results["target_delta_120m"].to_numpy(); p=blind_results["pred_delta_120m"].to_numpy(); e=p-y
summary=pd.DataFrame([{
    "model":FINAL_MODEL_KIND,"n":len(y),"MAE_cm":100*np.abs(e).mean(),"RMSE_cm":100*np.sqrt(np.mean(e**2)),
    "bias_cm":100*e.mean(),"median_abs_error_cm":100*np.median(np.abs(e)),"max_abs_error_cm":100*np.abs(e).max(),
    "within_20cm_pct":100*np.mean(np.abs(e)<=.2),"within_50cm_pct":100*np.mean(np.abs(e)<=.5),
}])
summary.to_csv(TABLES/"blind_2025_final_metrics_v6.csv",index=False)
display(summary)

blind_results["target_time"]=pd.to_datetime(blind_results["prediction_time"])+pd.Timedelta(minutes=120)
blind_results["real_delta_cm"]=100*blind_results["target_delta_120m"]
blind_results["pred_delta_cm"]=100*blind_results["pred_delta_120m"]
blind_results["real_future_stage"]=blind_results["stage_now"]+blind_results["target_delta_120m"]
blind_results["pred_future_stage"]=blind_results["stage_now"]+blind_results["pred_delta_120m"]
blind_results["error_cm"]=100*e
blind_results.to_csv(PREDICTIONS/"blind_2025_predictions_final_v6.csv",index=False)

In [ ]:
# Desempenho por severidade
bins=[-np.inf,0,.25,.5,1,2,3,np.inf]; labels=["<=0","0-0.25","0.25-0.5","0.5-1","1-2","2-3",">3"]
z=blind_results.copy(); z["band"]=pd.cut(z["target_delta_120m"],bins=bins,labels=labels)
rows=[]
for band in labels:
    q=z[z["band"]==band]
    if not len(q): continue
    er=q["pred_delta_120m"]-q["target_delta_120m"]
    rows.append({"severity_band":band,"n":len(q),"MAE_cm":100*er.abs().mean(),"RMSE_cm":100*np.sqrt(np.mean(er**2)),"bias_cm":100*er.mean(),"underprediction_pct":100*np.mean(er<0)})
sev=pd.DataFrame(rows); sev.to_csv(TABLES/"blind_2025_severity_v6.csv",index=False); display(sev)

In [ ]:
# Para redes temporais, avalia as tres heads de severidade com limiares congelados em 2023.
if alert_probabilities is not None:
    from utils.temporal_models import classification_metrics
    rows=[]
    for th_m,col in [(1.0,"p_ge_1m"),(2.0,"p_ge_2m"),(3.0,"p_ge_3m")]:
        m=classification_metrics(y>=th_m,alert_probabilities[col].to_numpy(),alert_thresholds[th_m])
        rows.append({"event_threshold_m":th_m,"probability_threshold":alert_thresholds[th_m],**m})
    alert_final=pd.DataFrame(rows); alert_final.to_csv(TABLES/"blind_2025_alerts_v6.csv",index=False); display(alert_final)

In [ ]:
# Metadados para o notebook operacional 09.
metadata = {
    "model_kind": FINAL_MODEL_KIND,
    "horizon_min": 120,
    "created_after_blind_test": True,
    "predictions_file": "outputs/predictions/blind_2025_predictions_final_v6.csv",
}
if FINAL_MODEL_KIND in ["TCN", "GRU"]:
    metadata["lookback_steps"] = lookback
    metadata["preprocessor"] = f"models/final_{FINAL_MODEL_KIND.lower()}_preprocessor.json"
    metadata["checkpoint"] = f"models/final_{FINAL_MODEL_KIND.lower()}_120m.pt"
    metadata["alert_thresholds"] = {str(k): v for k,v in alert_thresholds.items()}
else:
    metadata["model_file"] = "models/final_xgboost_radar_120m.json"
(MODELS / "final_model_metadata_v6.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Metadados salvos em models/final_model_metadata_v6.json")

## Regra depois deste notebook

Depois de olhar estas tabelas, 2025 deixou de ser desconhecido. Voce pode naturalmente melhorar um modelo de producao no futuro, inclusive usando 2025 no treinamento, mas qualquer nova escolha orientada pelo resultado deste holdout deve ser reportada como desenvolvimento posterior, nao como uma nova avaliacao cega no mesmo periodo.